## Imports and Setup

In [ ]:
import re
import csv
import math
import pprint
import requests
import pandas as pd
import pdfplumber
from docx import Document
from docx.oxml.ns import qn
from collections import Counter

import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

## Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

### (a) Extract text from each page of a PDF

In [ ]:
pdf_path = "/home/arham/test.pdf"  

pages_text = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text is None:
            text = ""
        pages_text.append(text)

print("Total number of pages extracted:", len(pages_text))

### (b) Merge pages and tokenize

In [ ]:
full_text = " ".join(pages_text)

cleaned_text = re.sub(r'[^a-zA-Z\s]', '', full_text)
cleaned_text = cleaned_text.lower()

tokens = cleaned_text.split()

print("Total token count:", len(tokens))

### (c) Word frequency dictionary (top 10 words)

In [ ]:
stopwords_list = [
    "the", "is", "in", "and", "to", "of", "a", "for", "on", "with",
    "as", "by", "at", "an", "be", "this", "that", "it", "from", "are"
]

filtered_tokens = [word for word in tokens if word not in stopwords_list]

word_freq = {}
for word in filtered_tokens:
    word_freq[word] = word_freq.get(word, 0) + 1

top_10_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 most frequent words:")
for word, count in top_10_words:
    print(f"{word}: {count}")

### (d) Search for a keyword and print matching sentences with page numbers

In [ ]:
keyword = input("Enter a keyword to search: ")

found_match = False
for page_num, page_text in enumerate(pages_text, start=1):
    sentences = re.split(r'(?<=[.!?])\s+', page_text)
    for sentence in sentences:
        if keyword.lower() in sentence.lower():
            print(f"Page {page_num}: {sentence.strip()}")
            found_match = True

if not found_match:
    print(f"No sentence containing '{keyword}' was found.")

## Task 2: Structured Page Metadata Extraction and CSV Export

### (a) Build page metadata dictionaries

In [ ]:
pdf_path = "/home/arham/test.pdf"  

pages_data = []
with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        raw_text = page.extract_text()
        if raw_text is None:
            raw_text = ""
        word_count = len(raw_text.split())
        char_count = len(raw_text.replace(" ", ""))
        pages_data.append({
            "page_number": i,
            "raw_text": raw_text,
            "word_count": word_count,
            "char_count": char_count
        })

print("First 2 entries of pages_data:")
for entry in pages_data[:2]:
    print(entry)

### (b) Highest / lowest word count pages and the average

In [ ]:
highest_page = max(pages_data, key=lambda p: p["word_count"])
nonzero_pages = [p for p in pages_data if p["word_count"] > 0]
lowest_page = min(nonzero_pages, key=lambda p: p["word_count"]) if nonzero_pages else None

total_words = sum(p["word_count"] for p in pages_data)
average_word_count = round(total_words / len(pages_data), 2) if pages_data else 0

print("Page with highest word count:")
print(f"Page {highest_page['page_number']} - {highest_page['word_count']} words")
print(highest_page["raw_text"])

print("\nPage with lowest non-zero word count:")
if lowest_page:
    print(f"Page {lowest_page['page_number']} - {lowest_page['word_count']} words")
else:
    print("No page has a non-zero word count.")

print("\nAverage word count across all pages:", average_word_count)

### (c) Case-insensitive phrase search with a custom exception

In [ ]:
class PhraseNotFoundError(Exception):
    pass

search_phrase = input("Enter a search phrase: ")

matches = []
for page in pages_data:
    count = page["raw_text"].lower().count(search_phrase.lower())
    if count > 0:
        matches.append((page["page_number"], count))

try:
    if not matches:
        raise PhraseNotFoundError(f"The phrase '{search_phrase}' was not found in any page.")
    for page_number, count in matches:
        print(f"Page {page_number}: '{search_phrase}' appears {count} time(s)")
except PhraseNotFoundError as e:
    print("Error:", e)

### (d) Write and re-read `pdf_summary.csv`

In [ ]:
with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for page in pages_data:
        writer.writerow([page["page_number"], page["word_count"], page["char_count"]])

print("pdf_summary.csv written successfully.\n")

print("Contents of pdf_summary.csv:")
with open("pdf_summary.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

## Task 3: PDF Text Preprocessing Pipeline for NLP

### (a) Extract text and clean it (remove URLs, digits, non-alphabetic characters)

In [ ]:
pdf_path = "sample.pdf"  # update this path

full_text = ""
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            full_text += text + " "

char_count_before = len(full_text)

cleaned = re.sub(r'http\S+|www\.\S+', '', full_text)          # remove URLs
cleaned = re.sub(r'\d+', '', cleaned)                           # remove digits
cleaned = re.sub(r'[^a-zA-Z\s.]', '', cleaned)                  # keep letters, spaces, full stops
cleaned = re.sub(r'\s+', ' ', cleaned).strip()                  # collapse multiple spaces

char_count_after = len(cleaned)

print("Character count before cleaning:", char_count_before)
print("Character count after cleaning:", char_count_after)

### (b) Lowercase, split into sentences, remove empty strings

In [ ]:
cleaned_lower = cleaned.lower()

sentences = re.split(r'[.!?]', cleaned_lower)
sentences = [s.strip() for s in sentences]

sentence_count_before = len(sentences)

sentences = [s for s in sentences if s != ""]

sentence_count_after = len(sentences)

print("Sentence count before removing empty strings:", sentence_count_before)
print("Sentence count after removing empty strings:", sentence_count_after)

### (c) Tokenize, remove stopwords, and filter short sentences

In [ ]:
english_stopwords = set(stopwords.words('english'))

sentence_count_before_filter = len(sentences)

filtered_sentences = []
for sentence in sentences:
    words = sentence.split()
    words_no_stopwords = [w for w in words if w not in english_stopwords]
    if len(words_no_stopwords) >= 5:
        filtered_sentences.append(sentence)

sentence_count_after_filter = len(filtered_sentences)

print("Sentence count before stopword filter:", sentence_count_before_filter)
print("Sentence count after stopword filter:", sentence_count_after_filter)

print("\n3 sample sentences:")
for s in filtered_sentences[:3]:
    print("-", s)

### (d) Write to `nlp_ready_output.txt` and analyze it

In [ ]:
with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    for sentence in filtered_sentences:
        f.write(sentence + "\n")

with open("nlp_ready_output.txt", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f.readlines()]

longest_sentence = max(lines, key=len) if lines else ""
shortest_sentence = min(lines, key=len) if lines else ""

print("Total lines written:", len(lines))
print("\nLongest sentence:", longest_sentence)
print("\nShortest sentence:", shortest_sentence)

## Task 4: DOCX Paragraph Classification and Statistical Analysis

### (a) Extract paragraph text and style names

In [ ]:
docx_path = "sample.docx"  # update this path

doc = Document(docx_path)

paragraphs_info = []
for para in doc.paragraphs:
    paragraphs_info.append({
        "text": para.text,
        "style": para.style.name
    })

unique_styles = set(p["style"] for p in paragraphs_info)

print("Total paragraph count:", len(paragraphs_info))
print("Unique style names found:", unique_styles)

### (b) Classify paragraphs into Headings, Body, and Other/Empty

In [ ]:
headings = []
body = []
other = []

for p in paragraphs_info:
    style = p["style"]
    if style.startswith("Heading"):
        headings.append(p)
    elif style == "Normal" or "Body" in style:
        body.append(p)
    else:
        other.append(p)

print("Headings count:", len(headings))
print("Body count:", len(body))
print("Other/Empty count:", len(other))

### (c) Statistics for Body paragraphs with more than 5 words

In [ ]:
qualifying_body = []
for p in body:
    word_count = len(p["text"].split())
    if word_count > 5:
        char_count = len(p["text"].replace(" ", ""))
        qualifying_body.append({
            "text": p["text"],
            "word_count": word_count,
            "char_count": char_count
        })

if qualifying_body:
    highest = max(qualifying_body, key=lambda x: x["word_count"])
    lowest = min(qualifying_body, key=lambda x: x["word_count"])
    mean_word_count = round(sum(p["word_count"] for p in qualifying_body) / len(qualifying_body), 2)

    print("Paragraph with highest word count:")
    print(highest["text"], "->", highest["word_count"], "words")

    print("\nParagraph with lowest word count (above 5):")
    print(lowest["text"], "->", lowest["word_count"], "words")

    print("\nMean word count:", mean_word_count)
else:
    print("No body paragraphs with more than 5 words were found.")

### (d) Nested dictionary of styles, sorted by count

In [ ]:
style_dict = {}
for p in paragraphs_info:
    style = p["style"]
    if style not in style_dict:
        style_dict[style] = {"count": 0, "texts": []}
    style_dict[style]["count"] += 1
    style_dict[style]["texts"].append(p["text"])

for style in style_dict:
    if style_dict[style]["count"] > 2:
        style_dict[style]["texts"] = style_dict[style]["texts"][:2]

sorted_style_dict = dict(
    sorted(style_dict.items(), key=lambda x: x[1]["count"], reverse=True)
)

pprint.pprint(sorted_style_dict)

## Task 5: DOCX Table Extraction and Pandas Integration

### (a) Extract all tables from the document

In [ ]:
docx_path = "sample_with_table.docx"  # update this path (must contain at least one table)

doc = Document(docx_path)

all_tables = []
for table in doc.tables:
    table_data = []
    for row in table.rows:
        row_data = [cell.text for cell in row.cells]
        table_data.append(row_data)
    all_tables.append(table_data)

print("Total number of tables found:", len(all_tables))
for i, table_data in enumerate(all_tables, start=1):
    rows = len(table_data)
    cols = len(table_data[0]) if rows > 0 else 0
    print(f"Table {i}: {rows} rows x {cols} columns")

### (b) Convert the first table into a pandas DataFrame

In [ ]:
first_table = all_tables[0]
header = first_table[0]
data_rows = first_table[1:]

df = pd.DataFrame(data_rows, columns=header)
print(df)

print("\nEmpty or None cells per column:")
empty_counts = df.apply(lambda col: col.isnull().sum() + (col == "").sum())
print(empty_counts)

### (c) Extract paragraphs that appear before the first table

In [ ]:
pre_table_paragraphs = []
for child in doc.element.body.iterchildren():
    if child.tag == qn('w:tbl'):
        break
    if child.tag == qn('w:p'):
        text = "".join(node.text for node in child.iter(qn('w:t')) if node.text)
        pre_table_paragraphs.append(text)

print("Number of pre-table paragraphs:", len(pre_table_paragraphs))

### (d) Write pre-table paragraphs and table data to `docx_extracted.txt`

In [ ]:
with open("docx_extracted.txt", "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for para in pre_table_paragraphs:
        f.write(para + "\n")

    f.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        f.write(" | ".join(row) + "\n")

print("docx_extracted.txt written successfully.")

## Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

### (a) Compare headings between two documents

In [ ]:
docx_path_a = "document_a.docx"  # update this path
docx_path_b = "document_b.docx"  # update this path

doc_a = Document(docx_path_a)
doc_b = Document(docx_path_b)

headings_a = [p.text for p in doc_a.paragraphs if p.style.name.startswith("Heading")]
headings_b = [p.text for p in doc_b.paragraphs if p.style.name.startswith("Heading")]

set_a = set(headings_a)
set_b = set(headings_b)

print("Headings common to both documents:", set_a & set_b)
print("Headings only in Document A:", set_a - set_b)
print("Headings only in Document B:", set_b - set_a)

### (b) Merge Normal paragraphs and remove duplicates

In [ ]:
normal_a = [p.text for p in doc_a.paragraphs if p.style.name == "Normal" and p.text.strip() != ""]
normal_b = [p.text for p in doc_b.paragraphs if p.style.name == "Normal" and p.text.strip() != ""]

merged_paragraphs = normal_a + normal_b
count_before = len(merged_paragraphs)

seen = set()
deduplicated_paragraphs = []
for para in merged_paragraphs:
    key = para.strip().lower()
    if key not in seen:
        seen.add(key)
        deduplicated_paragraphs.append(para)

count_after = len(deduplicated_paragraphs)

print("Total paragraphs before deduplication:", count_before)
print("Total paragraphs after deduplication:", count_after)

### (c) Combined word frequency (top 20 words)

In [ ]:
stopwords_15 = [
    "the", "is", "in", "and", "to", "of", "a", "for", "on", "with",
    "as", "by", "at", "an", "be"
]

all_words = []
for para in deduplicated_paragraphs:
    cleaned_para = re.sub(r'[^a-zA-Z\s]', '', para).lower()
    words = cleaned_para.split()
    all_words.extend(words)

filtered_words = [w for w in all_words if w not in stopwords_15]

word_freq = Counter(filtered_words)
top_20_words = word_freq.most_common(20)

print("Top 20 most frequent words:")
for word, count in top_20_words:
    print(f"{word}: {count}")

### (d) Summary report

In [ ]:
total_headings = len(headings_a) + len(headings_b)
total_unique_body_paragraphs = len(deduplicated_paragraphs)
vocabulary_size = len(set(filtered_words))

paragraph_lengths = [len(p.split()) for p in deduplicated_paragraphs]
average_paragraph_length = round(sum(paragraph_lengths) / len(paragraph_lengths), 2) if paragraph_lengths else 0

longest_paragraph = max(deduplicated_paragraphs, key=lambda p: len(p.split())) if deduplicated_paragraphs else ""

print("Total headings across both documents:", total_headings)
print("Total unique body paragraphs after deduplication:", total_unique_body_paragraphs)
print("Total unique word types (vocabulary size):", vocabulary_size)
print("Average paragraph length in words:", average_paragraph_length)
print("Longest paragraph by word count:")
print(longest_paragraph)

## Task 7: Multi-Endpoint API Data Joining and Analysis

### (a) Fetch users and posts, validating the status code

In [ ]:
users_url = "https://jsonplaceholder.typicode.com/users"
posts_url = "https://jsonplaceholder.typicode.com/posts"

users_response = requests.get(users_url)
if users_response.status_code != 200:
    raise ConnectionError(f"Failed to fetch users. Status code: {users_response.status_code}")
users_data = users_response.json()

posts_response = requests.get(posts_url)
if posts_response.status_code != 200:
    raise ConnectionError(f"Failed to fetch posts. Status code: {posts_response.status_code}")
posts_data = posts_response.json()

print("Users fetched:", len(users_data))
print("Posts fetched:", len(posts_data))

### (b) Join users with their posts

In [ ]:
joined_data = {}
for user in users_data:
    user_posts = [post["title"] for post in posts_data if post["userId"] == user["id"]]
    joined_data[user["name"]] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": user_posts
    }

print("Joined data for the first 3 users:")
for name in list(joined_data.keys())[:3]:
    print(name, "->", joined_data[name])

### (c) User with the most posts

In [ ]:
top_user_name = max(joined_data, key=lambda name: joined_data[name]["post_count"])
top_user_info = joined_data[top_user_name]

top_user_full = next(u for u in users_data if u["name"] == top_user_name)

print("Full name:", top_user_name)
print("Email:", top_user_info["email"])
print("Company name:", top_user_full["company"]["name"])
print("\nAll post titles:")
for i, title in enumerate(top_user_info["titles"], start=1):
    print(f"{i}. {title}")

### (d) Filter users by company buzzword or long city name

In [ ]:
matching_users = []
for user in users_data:
    bs_text = user["company"]["bs"].lower()
    city = user["address"]["city"]
    if "synergies" in bs_text or len(city) > 8:
        post_count = joined_data[user["name"]]["post_count"]
        matching_users.append((user["name"], city, post_count))

print("Users matching the filter conditions:")
for name, city, post_count in matching_users:
    print(f"Name: {name}, City: {city}, Post Count: {post_count}")

## Task 8: To-Do Completion Rate Analysis Across Users

### (a) Fetch and validate the todos data

In [ ]:
todos_url = "https://jsonplaceholder.typicode.com/todos"

response = requests.get(todos_url)

if response.status_code != 200:
    print("Error: could not fetch todos. Status code:", response.status_code)
else:
    todos_data = response.json()
    if not isinstance(todos_data, list):
        print("Error: response is not a list.")
    elif len(todos_data) == 0:
        print("Error: no todo records found.")
    else:
        print("Total number of todo records retrieved:", len(todos_data))

### (b) Group todos by user and verify totals

In [ ]:
user_todo_summary = {}

for todo in todos_data:
    uid = todo["userId"]
    if uid not in user_todo_summary:
        user_todo_summary[uid] = {"total": 0, "completed": 0, "incomplete": 0}
    user_todo_summary[uid]["total"] += 1
    if todo["completed"]:
        user_todo_summary[uid]["completed"] += 1
    else:
        user_todo_summary[uid]["incomplete"] += 1

for uid in sorted(user_todo_summary.keys()):
    stats = user_todo_summary[uid]
    check = (stats["completed"] + stats["incomplete"]) == stats["total"]
    print(f"User {uid}: {stats} -> Valid: {check}")

### (c) Completion rate per user, highest/lowest, and overall average

In [ ]:
completion_rates = {}
for uid, stats in user_todo_summary.items():
    rate = round((stats["completed"] / stats["total"]) * 100, 1)
    completion_rates[uid] = rate

highest_uid = max(completion_rates, key=completion_rates.get)
lowest_uid = min(completion_rates, key=completion_rates.get)
overall_average = round(sum(completion_rates.values()) / len(completion_rates), 1)

print("Highest completion rate:", f"User {highest_uid} -> {completion_rates[highest_uid]}%")
print("Lowest completion rate:", f"User {lowest_uid} -> {completion_rates[lowest_uid]}%")
print("Overall average completion rate:", f"{overall_average}%")

### (d) Ranked table of users by completion rate

In [ ]:
users_response = requests.get("https://jsonplaceholder.typicode.com/users")
users_lookup_data = users_response.json()

user_lookup = {user["id"]: user["name"] for user in users_lookup_data}

table_rows = []
for uid, stats in user_todo_summary.items():
    table_rows.append({
        "userId": uid,
        "name": user_lookup.get(uid, "Unknown"),
        "total": stats["total"],
        "completed": stats["completed"],
        "rate": completion_rates[uid]
    })

table_rows.sort(key=lambda x: x["rate"], reverse=True)

print(f"{'Rank':<6}{'Name':<20}{'Total Tasks':<14}{'Completed':<12}{'Completion Rate %':<18}")
for rank, row in enumerate(table_rows, start=1):
    print(f"{rank:<6}{row['name']:<20}{row['total']:<14}{row['completed']:<12}{row['rate']:<18}")

## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

### (a) Flatten nested user objects

In [ ]:
users_response = requests.get("https://jsonplaceholder.typicode.com/users")
users_data = users_response.json()

def flatten_user(user):
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"]
    }

flattened_users = [flatten_user(user) for user in users_data]

print("Flattened user list:")
for user in flattened_users:
    print(user)

### (b) DataFrame operations and distance from origin

In [ ]:
df_users = pd.DataFrame(flattened_users)
print(df_users)

df_users["lat"] = df_users["lat"].astype(float)
df_users["lng"] = df_users["lng"].astype(float)

df_users["distance_from_origin"] = df_users.apply(
    lambda row: math.sqrt(row["lat"] ** 2 + row["lng"] ** 2), axis=1
)

farthest_user = df_users.loc[df_users["distance_from_origin"].idxmax()]
print("\nUser with the greatest distance from origin:")
print(farthest_user)

### (c) Northern vs Southern Hemisphere comparison

In [ ]:
northern_users = df_users[df_users["lat"] > 0]["name"].tolist()
southern_users = df_users[df_users["lat"] < 0]["name"].tolist()

print("Northern Hemisphere users:", northern_users)
print("Southern Hemisphere users:", southern_users)

northern_avg_distance = df_users[df_users["lat"] > 0]["distance_from_origin"].mean()
southern_avg_distance = df_users[df_users["lat"] < 0]["distance_from_origin"].mean()

if northern_avg_distance > southern_avg_distance:
    print("\nNorthern Hemisphere has the higher average distance from origin.")
else:
    print("\nSouthern Hemisphere has the higher average distance from origin.")

### (d) Comment counts per post — top 5 and bottom 5

In [ ]:
posts_response = requests.get("https://jsonplaceholder.typicode.com/posts")
posts_data = posts_response.json()

comments_response = requests.get("https://jsonplaceholder.typicode.com/comments")
comments_data = comments_response.json()

comment_counts = {}
for comment in comments_data:
    pid = comment["postId"]
    comment_counts[pid] = comment_counts.get(pid, 0) + 1

post_comment_list = []
for post in posts_data:
    pid = post["id"]
    count = comment_counts.get(pid, 0)
    post_comment_list.append((pid, post["title"][:40], count))

post_comment_list.sort(key=lambda x: x[2], reverse=True)

print("Top 5 most-commented posts:")
for pid, title, count in post_comment_list[:5]:
    print(f"Post {pid}: '{title}' -> {count} comments")

print("\nBottom 5 least-commented posts:")
for pid, title, count in post_comment_list[-5:]:
    print(f"Post {pid}: '{title}' -> {count} comments")